# VintageGAN Interactive Demo

This notebook provides an interactive demo of VintageGAN with sliders to control each defect parameter.

**Requirements:**
- Trained VintageGAN model
- ipywidgets for interactive sliders

**Install widgets:**
```bash
pip install ipywidgets
jupyter nbextension enable --py widgetsnbextension
```

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, Button, VBox, HBox
from IPython.display import display

from inference import VintageFilter

# Set matplotlib backend
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Model and Test Image

In [ ]:
# Initialize VintageFilter
CHECKPOINT_PATH = '../checkpoints/generator_final.pth'  # Update this path

print("Loading VintageGAN model...")
filter = VintageFilter(CHECKPOINT_PATH, device='cuda' if torch.cuda.is_available() else 'cpu')
print(f"Model loaded on: {filter.device}")

In [ ]:
# Load test image
# Option 1: Use your own image
IMAGE_PATH = '../data/test_image.jpg'  # Update this path

# Option 2: Create a sample colored image for testing
# test_img = Image.new('RGB', (512, 512), color=(100, 150, 200))

try:
    original_image = Image.open(IMAGE_PATH).convert('RGB')
    print(f"Loaded image: {original_image.size}")
except:
    print("Could not load image, creating sample image")
    original_image = Image.new('RGB', (512, 512), color=(120, 180, 220))

# Display original
plt.figure(figsize=(6, 6))
plt.imshow(original_image)
plt.title('Original Image')
plt.axis('off')
plt.show()

## 2. Interactive Vintage Effect Controller

Use the sliders below to adjust each defect parameter:

In [ ]:
def apply_vintage_effect(grain=0.5, scratch=0.3, dust=0.2, vignette=0.4, color_shift=0.5, blur=0.1):
    """
    Apply vintage effect with specified parameters.
    
    Args:
        grain: Film grain intensity [0-1]
        scratch: Scratch density [0-1]
        dust: Dust particle count [0-1]
        vignette: Vignetting strength [0-1]
        color_shift: Color degradation [0-1]
        blur: Blur amount [0-1]
    """
    # Create condition dictionary
    conditions = {
        'grain': grain,
        'scratch': scratch,
        'dust': dust,
        'vignette': vignette,
        'color_shift': color_shift,
        'blur': blur
    }
    
    # Apply filter
    vintage_image = filter.apply(original_image, conditions=conditions)
    
    # Display comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    
    axes[0].imshow(original_image)
    axes[0].set_title('Original', fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(vintage_image)
    axes[1].set_title('Vintage Effect Applied', fontsize=14)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print condition values
    print("\nCurrent Settings:")
    for key, val in conditions.items():
        bar = '█' * int(val * 20) + '░' * (20 - int(val * 20))
        print(f"  {key:12s}: {bar} {val:.2f}")

# Create interactive widget
interact(
    apply_vintage_effect,
    grain=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description='Grain:', continuous_update=False),
    scratch=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.3, description='Scratch:', continuous_update=False),
    dust=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.2, description='Dust:', continuous_update=False),
    vignette=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.4, description='Vignette:', continuous_update=False),
    color_shift=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description='Color Shift:', continuous_update=False),
    blur=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.1, description='Blur:', continuous_update=False)
);

## 3. Preset Gallery

Try different vintage presets:

In [ ]:
# Available presets
PRESETS = ['light', 'medium', 'heavy', 'grain_only', 'faded', 'scratched']

# Generate preset gallery
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, preset in enumerate(PRESETS):
    vintage_image = filter.apply(original_image, conditions=preset)
    axes[idx].imshow(vintage_image)
    axes[idx].set_title(f'Preset: {preset}', fontsize=12, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 4. Progressive Defect Visualization

See how each individual defect affects the image:

In [ ]:
# Test each defect individually
defect_types = ['grain', 'scratch', 'dust', 'vignette', 'color_shift', 'blur']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, defect in enumerate(defect_types):
    # Create condition with only this defect at 0.7
    conditions = {key: 0.0 for key in defect_types}
    conditions[defect] = 0.7
    
    vintage_image = filter.apply(original_image, conditions=conditions)
    axes[idx].imshow(vintage_image)
    axes[idx].set_title(f'{defect.replace("_", " ").title()} Only (0.7)', fontsize=12)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 5. Intensity Progression

See how increasing intensity affects the output:

In [ ]:
# Show progression for one defect type
defect = 'grain'  # Change this to test different defects
intensities = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for idx, intensity in enumerate(intensities):
    conditions = {key: 0.0 for key in defect_types}
    conditions[defect] = intensity
    
    vintage_image = filter.apply(original_image, conditions=conditions)
    axes[idx].imshow(vintage_image)
    axes[idx].set_title(f'{defect}: {intensity:.2f}', fontsize=11)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 6. Save Your Results

Save the vintage image with your custom settings:

In [ ]:
# Define your custom conditions
custom_conditions = {
    'grain': 0.6,
    'scratch': 0.4,
    'dust': 0.3,
    'vignette': 0.5,
    'color_shift': 0.7,
    'blur': 0.2
}

# Apply and save
output_image = filter.apply(original_image, conditions=custom_conditions)
output_path = '../outputs/vintage_output.jpg'

# Create output directory if needed
import os
os.makedirs('../outputs', exist_ok=True)

output_image.save(output_path, quality=95)
print(f"✅ Saved vintage image to: {output_path}")

# Display result
plt.figure(figsize=(8, 8))
plt.imshow(output_image)
plt.title('Saved Vintage Image', fontsize=14)
plt.axis('off')
plt.show()

## 7. Tips for Best Results

**Realistic Vintage Look:**
- Use medium grain (0.4-0.6)
- Light scratches (0.2-0.4)
- Subtle vignetting (0.3-0.5)
- Moderate color shift (0.5-0.7)
- Minimal blur (0.1-0.2)

**Heavy Aged Look:**
- High grain (0.7-0.9)
- More scratches (0.5-0.7)
- More dust (0.4-0.6)
- Strong vignetting (0.6-0.8)
- Heavy color shift (0.7-0.9)

**Subtle Enhancement:**
- Light grain (0.2-0.3)
- No scratches (0.0)
- Minimal dust (0.1)
- Light vignetting (0.2-0.3)
- Slight color shift (0.3-0.4)